In [1]:
"""
Objective: Improve on an existing prompt by using the prompt engineering topics discussed in the Prompt Engineering Techniques section
"""

# Identify changes in source files and reload
%load_ext autoreload
%autoreload 2

# Import API key and Anthropic API
from dotenv import load_dotenv
from anthropic import Anthropic

import json

"""
Import Helper functions for:
- Chat
- Dataset generation
"""
from claude_chat import ClaudeChat
from claude_dataset import ClaudeDataset
from claude_evaluation import ClaudeEvaluation

In [2]:
# Access the API key
load_dotenv()

# Access the Anthropic API
client = Anthropic()

# Specify the model Claude will use
model = "claude-sonnet-5"

"""
Access:
- Conversation history related to scholarly article topics
- Functions to store user inputs
- Functions to store Claude responses
"""
claude_chat = ClaudeChat(model, client)

"""
Access:
- Conversation history related to generating dataset JSON data
- Functions to store dataset JSON data in a JSON file
- Functiosn to load dataset JSON data from a JSON file
"""
generated_dataset = ClaudeDataset(model, client)

"""
Access:
- Functions to test a prompt against a dataset
- Functions to utilize model based grading using Claude
- Functions to utilize code based grading
- Functions to calculate the average score
"""
claude_evaluation = ClaudeEvaluation(client, model)

In [3]:
"""
Prompt engineering rules to replace prefilling
Rules for generating the JSON list of scholarly article topics
"""
scholarly_topics_prompt_rules = """
Rules:
- Return only valid JSON
- Do not use markdown
- Do not include comments
- Do not include explanations
- After the plain text, write END_OF_COMMANDS
"""

dataset_generation_prompt_rules = """
Rules:
- Return only valid JSON
- Do not use markdown
- Do not include comments
- Do not include explanations outside the JSON
- Do not include trailing commas
- After the JSON, write END_OF_COMMANDS
"""

# Provide Claude with context to customize how Claude responds to user inputs
evaluation_system_prompt = """
You are an expert at analyzing passages of scholarly text and extracting all the main topics from the text
"""

claude_chat.stop_sequences.append("END_OF_COMMANDS")

In [4]:
# Load in sample scholarly text
f = open("short_scholarly_text.txt")
scholarly_text = f.read()

In [5]:
# Prompting Process: 1. Create a draft of the prompt
# Restricted prompt to prevent max_token issues
prompt_draft = "What 3 topics are in here? Use very short sentences"

prompt_draft_extended = f"""
{prompt_draft}

{scholarly_text}

{scholarly_topics_prompt_rules}
"""

claude_chat.userInput(prompt_draft_extended)

claude_response = claude_chat.askClaude(evaluation_system_prompt)

In [6]:
# Parse just to see the output better
parsed_response = json.loads(claude_response)

# Observe the output from the initial prompt draft
print(json.dumps(parsed_response, indent=2))

{
  "topics": [
    {
      "title": "Awareness over default self-centeredness",
      "description": "Humans default to seeing themselves as the center. Real thinking means choosing awareness. This breaks automatic self-focus."
    },
    {
      "title": "Liberal arts education as learning to choose thought",
      "description": "Education is not just knowledge. It teaches control over thinking. It means choosing meaning, not just absorbing facts."
    },
    {
      "title": "Worship and freedom in daily adult life",
      "description": "Everyone worships something by default. Choosing what to worship matters greatly. True freedom is conscious care for others, found in mundane routines."
    }
  ]
}


In [10]:
# Prompting Process: 2. Generate test data
dataset_prompt = f"""
Generate a small evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that extract all the topics into a JSON array of strings. The topics focus on what the article talks about. Generate an array of JSON objects, each representing a paragraph from a scholarly journal written in English.

Example output:
[
	{{
		"content": "A very short scholarly-style paragraph between 30 and 60 words.",
        "format": "text",
        "solution_criteria": "A very short key criteria for evaluating the solution between 20 and 40 words."
	}},
    ...
]
 
 Please generate 3 objects
 
 {dataset_generation_prompt_rules}
"""

# Generate the dataset and store it in dataset.json
generated_dataset.generateDataset(dataset_prompt)

In [11]:
# Observe the output
dataset = json.dumps(generated_dataset.getEvaluationDataset(), indent=2)

print(dataset)

[
  {
    "content": "Recent advances in CRISPR-Cas9 gene editing have enabled precise modifications to plant genomes, allowing researchers to enhance drought resistance in staple crops such as wheat and maize. These developments hold significant promise for improving global food security amid changing climate conditions.",
    "format": "text",
    "solution_criteria": "Should include topics such as CRISPR-Cas9, gene editing, drought resistance, crop improvement, food security, and climate change."
  },
  {
    "content": "The rise of social media platforms has transformed political discourse, enabling rapid dissemination of information but also facilitating the spread of misinformation. This paper examines how algorithmic curation influences public opinion formation and contributes to increasing polarization among voters.",
    "format": "text",
    "solution_criteria": "Should include topics such as social media, political discourse, misinformation, algorithmic curation, public opin

In [12]:
# Prompting Process: 3. Evaluate the prompt

# Use the Claude generated dataset
dataset = generated_dataset.getEvaluationDataset()

# Run the dataset against the initial prompt draft
dataset_prompt_results = claude_evaluation.testPrompt(dataset, prompt_draft, dataset_generation_prompt_rules)

In [14]:
# Use model based grading to evaluate Claude's responses to the initial prompt draft
model_grading_results = claude_evaluation.modelBasedGrading(dataset, dataset_prompt_results, dataset_generation_prompt_rules)

In [18]:
# Use code based grading to verify the generated code has valid syntax and follows the correct format
code_grading_results = claude_evaluation.codeBasedGrading(dataset_prompt_results)

In [20]:
# Calcuate the mean of the model based grading and the code based grading
mean = claude_evaluation.calculateAverage(model_grading_results, code_grading_results)

In [21]:
# Observe the grading values
print("Model Based Grading Results")
print(json.dumps(model_grading_results, indent=2))

# Observe the grading values
print("Code Based Grading Results")
print(json.dumps(code_grading_results, indent=2))

# Observe the average
print(f"Prompt Evaluation Average: {mean}")

Model Based Grading Results
[
  "{\n\"strengths\": [\"Covers all required topics concisely\", \"Logical grouping of related concepts\"],\n\"weaknesses\": [\"Merges distinct topics into fewer categories, reducing granularity\", \"No explicit mention of wheat/maize as staple crop examples\"],\n\"reasoning\": \"The solution addresses all required topics but condenses them broadly rather than distinctly.\",\n\"score\": 8\n}",
  "{\"strengths\": [\"Covers all required topics\", \"Concise and well-organized grouping\"], \"weaknesses\": [\"Merges distinct topics into fewer entries\", \"Could separate polarization and public opinion more explicitly\"], \"reasoning\": \"The solution addresses all criteria topics but consolidates them broadly rather than distinctly.\", \"score\": 8}",
  "{\"strengths\": [\"Covers all required topics\", \"Concise phrasing\"], \"weaknesses\": [\"Merges distinct topics into few bullets\"], \"reasoning\": \"All required topics are present though slightly condensed.\

In [26]:
# Prompting Process: 4. Revise The Prompt
"""
Utilize XML tags to add structure and clarity to the prompt
Utilize examples to give Claude sample inputs and outputs to guide its response
"""

with open("I Have A Dream.txt", "r", encoding="utf-8") as f:
    mlk_speech = f.read()

revised_prompt = f"""
Provide a list of the main topics in this scholarly text.
Keep each topic less than 60 words

<scholarly_text>
{scholarly_text}
</scholarly_text>

Here is an example input with an ideal response
<sample_input>
{mlk_speech}
</sample_input>

<idea_output>
[
    "Racial equality",
    "Freedom and justice",
    "Unity and hope"
</ideal_output>
]

{scholarly_topics_prompt_rules}
"""

# Start with a fresh chat to observe the improvements
claude_chat.clearChatHistory()

# Store the revised prompt in the conversation history
claude_chat.userInput(revised_prompt)

claude_response = claude_chat.askClaude()

# Store Claude's response in the conversation history
claude_chat.claudeResponse(claude_response)

In [30]:
# Parse just to see the output better
parsed_response = json.loads(claude_response)

# Observe the output from the initial prompt draft
print(json.dumps(parsed_response, indent=2))

[
  "Awareness and consciousness",
  "Self-centeredness versus empathy",
  "Liberal arts education",
  "Freedom of thought",
  "Worship and meaning-making",
  "Everyday adult life and routine",
  "Choice and perception",
  "Default settings of the mind"
]


In [32]:
# Prompting Process: 5. Evaluate The Revised Prompt

# Run the dataset against the revised prompt draft
dataset_prompt_results = claude_evaluation.testPrompt(dataset, revised_prompt, dataset_generation_prompt_rules)

In [48]:

# Use model based grading to evaluate Claude's responses to the revised prompt draft
model_grading_results = claude_evaluation.modelBasedGrading(dataset, dataset_prompt_results, dataset_generation_prompt_rules)

In [50]:
# Use code based grading to verify the generated code has valid syntax and follows the correct format
code_grading_results = claude_evaluation.codeBasedGrading(dataset_prompt_results)

In [52]:
# Calcuate the mean of the model based grading and the code based grading
mean = claude_evaluation.calculateAverage(model_grading_results, code_grading_results)

In [53]:
# Observe the grading values
print("Model Based Grading Results")
print(json.dumps(model_grading_results, indent=2))

# Observe the grading values
print("Code Based Grading Results")
print(json.dumps(code_grading_results, indent=2))

# Observe the average
print(f"Prompt Evaluation Average: {mean}")

      lake taxo sc 

      little switzerland 226

SyntaxError: unterminated f-string literal (detected at line 10) (3827618771.py, line 10)